# 📚 SQL Ch.8 — BA-Specific Patterns
> BigQuery SQL Reference Guide, Chapter 8: MoM/YoY · Pareto Analysis · PIVOT · Cohort Retention · Deduplication · Data Quality Checks  
> BigQuery SQL 완전 참조 가이드 8장: MoM/YoY · 파레토 분석 · PIVOT · 코호트 리텐션 · 중복 제거 · 데이터 품질 체크

🎓 **This is the final chapter of the guide / 이 가이드의 마지막 챕터입니다.** Nothing here is a new keyword — every pattern below is Chapters 1–7's tools (`JOIN`, `GROUP BY`, `CTE`, `CASE WHEN`, window functions, date functions), combined and given a business name.  
여기엔 새로운 키워드가 하나도 없습니다 — 아래 모든 패턴은 1~7장의 도구(`JOIN`, `GROUP BY`, `CTE`, `CASE WHEN`, 윈도우 함수, 날짜 함수)를 조합해서 비즈니스 이름을 붙인 것입니다.

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배우고 싶은 것:
- [x] Build a MoM/YoY growth report and a Pareto (80/20) analysis by combining `LAG`, `DATE_DIFF`, and running-total window functions  
`LAG`, `DATE_DIFF`, 누적합 윈도우 함수를 조합해 MoM/YoY 성장률 리포트와 파레토(80/20) 분석을 만든다
- [x] Build a signup-cohort retention table by chaining CTEs together, each one operating on the previous stage's output  
CTE를 단계별로 이어 붙여, 각 단계가 이전 단계의 결과 위에서 동작하는 가입 코호트 리텐션 표를 만든다
- [x] Run a data-quality check (NULLs, duplicates, outliers) on any new table before trusting it for analysis  
어떤 새 테이블이든 분석에 신뢰하기 전에 데이터 품질 체크(NULL, 중복, 이상치)를 실행한다

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**EN:** This chapter isn't new syntax — it's the "greatest hits" of everything learned so far, assembled into the specific query shapes a BA writes on repeat: growth-over-time reports, 80/20 analyses, pivoted summary tables, retention cohorts, deduplication, and pre-analysis data quality checks. Recognizing these shapes means recognizing which tools to reach for the moment a stakeholder asks a familiar-sounding question.

**KR:** 이번 챕터는 새로운 문법이 아니라, 지금까지 배운 모든 것의 "베스트 히트"를 BA가 반복해서 작성하는 구체적인 쿼리 형태로 조립한 것입니다: 시간에 따른 성장 리포트, 80/20 분석, 피벗된 요약 표, 리텐션 코호트, 중복 제거, 분석 전 데이터 품질 체크. 이 형태들을 알아본다는 것은, 이해관계자가 익숙한 느낌의 질문을 던지는 순간 어떤 도구를 꺼내야 할지 알아본다는 뜻입니다.

## Why do we use it?
*(When is it useful?)*

**EN:** Business questions rarely map to one clean SQL keyword — "are we growing," "what drives 80% of our revenue," "are new users sticking around" all require *combining* several techniques into a single coherent query. Recognizing the pattern (not just the individual pieces) is what separates knowing SQL syntax from being able to actually answer business questions with it.

**KR:** 비즈니스 질문은 SQL 키워드 하나에 깔끔하게 대응하는 경우가 거의 없습니다 — "우리 성장하고 있나", "매출의 80%는 무엇이 만드나", "신규 유저가 계속 남아있나" 모두 여러 기법을 *조합*해야 하나의 일관된 쿼리가 됩니다. (개별 조각이 아니라) 패턴 자체를 알아보는 것이 SQL 문법을 아는 것과 실제로 비즈니스 질문에 답할 수 있는 것을 가르는 지점입니다.

## When is it used in Business Analytics?
*(Real-world use case)*

**EN:** Every single pattern in this chapter is something a working BA is asked for within their first month on the job: a growth report for a leadership meeting, a Pareto chart for inventory prioritization, a pivoted table for a spreadsheet-native stakeholder, a retention cohort for a product review, and — quietly, underneath everything — a data quality check run before any of it goes out the door.

**KR:** 이번 챕터의 모든 패턴은 실무 BA가 입사 첫 달 안에 요청받는 것들입니다: 리더십 미팅용 성장 리포트, 재고 우선순위를 위한 파레토 차트, 스프레드시트에 익숙한 이해관계자를 위한 피벗 표, 제품 리뷰를 위한 리텐션 코호트 — 그리고 조용히 그 모든 것 밑에서, 무엇이든 나가기 전에 실행하는 데이터 품질 체크.

**Comparison / 비교표:**

| BA Question / BA의 질문 | SQL Pattern | Pandas |
|---|---|---|
| Are we growing? / 성장하고 있나 | `LAG` MoM/YoY | `.pct_change()` |
| What drives 80% of revenue? / 매출의 80%는 무엇이 | Cumulative `SUM OVER` + Pareto | `.cumsum() / .sum()` |
| Month × region grid / 월×지역 그리드 | `CASE`+`SUM` crosstab | `pd.pivot_table()` |
| Are users sticking around? / 유저가 남아있나 | Cohort CTE chain | `groupby` + `unstack()` |
| Which row is the "real" one? / 어느 행이 "진짜"인가 | `ROW_NUMBER` + `QUALIFY` dedup | `.drop_duplicates(keep=...)` |
| Can I trust this table? / 이 테이블을 믿어도 되나 | NULL/dup/outlier audit | custom quality-check function |

---
# 📝 Syntax

## Basic Syntax
MoM (month-over-month) growth — `LAG` compares each row to the row directly before it.
MoM(전월 대비) 성장률 — `LAG`가 각 행을 바로 이전 행과 비교합니다.

In [1]:
# --- Environment setup / 환경 설정 ---
# We use DuckDB: a free, in-memory SQL engine that understands BigQuery-style syntax
# almost 1:1 (window functions, QUALIFY, ROLLUP, STRING_AGG, etc.), and can query
# pandas DataFrames directly by name -- no separate "load data" step needed.
# DuckDB는 무료 인메모리 SQL 엔진으로, BigQuery 문법(윈도우 함수, QUALIFY, ROLLUP,
# STRING_AGG 등)을 거의 그대로 이해하고, pandas DataFrame을 이름으로 바로 조회할 수
# 있습니다. 별도의 "데이터 로드" 단계가 필요 없습니다.
import duckdb
import pandas as pd
from IPython.display import display

def run(sql: str) -> pd.DataFrame:
    """Execute a SQL string against DuckDB and return the result as a DataFrame.
    SQL 문자열을 DuckDB에서 실행하고 결과를 DataFrame으로 반환합니다."""
    return duckdb.sql(sql).df()

duckdb.sql("CREATE OR REPLACE MACRO safe_divide(a, b) AS CASE WHEN b = 0 THEN NULL ELSE a / b END")

monthly_revenue = pd.DataFrame({
    "month":   ["2023-01","2023-02","2023-03","2024-01","2024-02","2024-03"],
    "revenue": [4800000, 5100000, 4600000, 5200000, 5800000, 5300000],
})

sql = """
SELECT
    month,
    revenue,
    LAG(revenue) OVER (ORDER BY month) AS prev_month,
    revenue - LAG(revenue) OVER (ORDER BY month) AS mom_diff,
    ROUND(SAFE_DIVIDE(
        revenue - LAG(revenue) OVER (ORDER BY month),
        LAG(revenue) OVER (ORDER BY month)
    ) * 100, 1) AS mom_pct
FROM monthly_revenue
ORDER BY month
"""
display(run(sql))
# ⚠️ Notice 2024-01's prev_month is 2023-03 (4,600,000) -- LAG just grabs "the row right before,"
# oblivious to the year boundary. This is fine for MoM (adjacent months truly are adjacent),
# but it's exactly why YoY (below) can't just reuse this same simple LAG(1) approach.
# ⚠️ 2024-01의 prev_month가 2023-03(4,600,000)임에 주목 -- LAG는 그냥 "바로 이전 행"을 가져올
# 뿐, 연도 경계는 신경 쓰지 않음. MoM에서는 문제없지만(인접한 달은 실제로 인접함),
# 바로 이 때문에 아래 YoY는 이 단순한 LAG(1) 방식을 그대로 재사용할 수 없음.


,month,revenue,prev_month,mom_diff,mom_pct
0,2023-01,4800000,<NA>,<NA>,NaN
1,2023-02,5100000,4800000,300000,6.3
2,2023-03,4600000,5100000,-500000,-9.8
3,2024-01,5200000,4600000,600000,13.0
4,2024-02,5800000,5200000,600000,11.5
5,2024-03,5300000,5800000,-500000,-8.6


## Common Variations

In [2]:
# YoY (year-over-year): LAG(col, 12) looks back 12 ROWS -- which only equals "12 months ago"
# if the data has zero gaps. Needs a full, consecutive 12+ months to work correctly.
# YoY(전년 동월 대비): LAG(col, 12)는 12 "행" 뒤를 봄 -- 데이터에 빈 달이 전혀 없어야만
# 이게 "12개월 전"과 같아짐. 정확히 동작하려면 빠짐없는 연속된 12개월 이상이 필요.
monthly_revenue_yoy = pd.DataFrame({
    "month": ["2023-01","2023-02","2023-03","2023-04","2023-05","2023-06",
              "2023-07","2023-08","2023-09","2023-10","2023-11","2023-12",
              "2024-01","2024-02","2024-03"],
    "revenue": [4800000,5100000,4600000,4900000,5000000,5200000,
                5400000,5300000,5100000,5000000,5600000,6200000,
                5200000,5800000,5300000],
})

sql = """
WITH monthly AS (
    SELECT month, revenue,
        LAG(revenue, 12) OVER (ORDER BY month) AS prev_year_revenue
    FROM monthly_revenue_yoy
)
SELECT month, revenue, prev_year_revenue,
    ROUND(SAFE_DIVIDE(revenue - prev_year_revenue, prev_year_revenue) * 100, 1) AS yoy_pct
FROM monthly
WHERE prev_year_revenue IS NOT NULL
ORDER BY month
"""
display(run(sql))
# Only 2024 rows show up -- 2023 rows have no "12 rows back" to compare against.
# If your real data has missing months, LAG(12) silently compares against the WRONG month --
# a Self JOIN + DATE_DIFF (safer, but more verbose) is the guide's recommended fix for that case.
# 2024년 행만 나옴 -- 2023년 행은 비교할 "12행 이전"이 없음.
# 실제 데이터에 빠진 달이 있다면 LAG(12)는 조용히 잘못된 달과 비교하게 됨 --
# Self JOIN + DATE_DIFF(더 안전하지만 더 장황함)가 이 경우 가이드가 권장하는 해결책.


,month,revenue,prev_year_revenue,yoy_pct
0,2024-01,5200000,4800000,8.3
1,2024-02,5800000,5100000,13.7
2,2024-03,5300000,4600000,15.2


---
# 🧪 Small Examples

## Example 1 — Cumulative Sum + Pareto Analysis / 누적합 + 파레토 분석
**EN:** The Pareto principle ("80% of revenue comes from 20% of products") is checked by sorting rows by size, computing a *running* total percentage, and seeing where it crosses 80%. It's the exact same `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` running-total pattern from Chapter 7, divided by the grand total.  
**KR:** 파레토 법칙("매출의 80%가 제품의 20%에서 나온다")은 행을 크기순으로 정렬하고 *누적* 비율을 계산해서 어디서 80%를 넘는지 보는 방식으로 확인합니다. 7장에서 배운 `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` 누적합 패턴을 전체 합계로 나눈 것과 정확히 같습니다.

In [3]:
product_revenue = pd.DataFrame({
    "product": ["노트북", "스마트폰", "태블릿", "키보드", "마우스", "이어폰", "충전기", "케이스"],
    "revenue": [36000000, 18000000, 9000000, 5400000, 4800000, 3600000, 2400000, 800000],
})

sql = """
WITH ranked AS (
    SELECT
        product,
        revenue,
        SUM(revenue) OVER () AS total_revenue,
        SUM(revenue) OVER (
            ORDER BY revenue DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_revenue
    FROM product_revenue
)
SELECT
    product,
    revenue,
    ROUND(revenue / total_revenue * 100, 1) AS rev_pct,
    ROUND(cumulative_revenue / total_revenue * 100, 1) AS cum_pct
FROM ranked
ORDER BY revenue DESC
"""
display(run(sql))
# Read it: cum_pct crosses 80% right around the 3rd-4th product -- just 노트북+스마트폰+태블릿
# (3 of 8 products) already account for ~79% of revenue. Those 3 deserve the inventory/marketing focus.
# 읽는 법: cum_pct가 80%를 넘는 지점이 3~4번째 제품 근처 -- 노트북+스마트폰+태블릿(8개 중 3개)만으로
# 이미 매출의 약 79%. 이 3개가 재고·마케팅 우선순위 대상.


,product,revenue,rev_pct,cum_pct
0,노트북,36000000,45.0,45.0
1,스마트폰,18000000,22.5,67.5
2,태블릿,9000000,11.3,78.8
3,키보드,5400000,6.8,85.5
4,마우스,4800000,6.0,91.5
5,이어폰,3600000,4.5,96.0
6,충전기,2400000,3.0,99.0
7,케이스,800000,1.0,100.0


## Example 2 — PIVOT Simulation: CASE+SUM Crosstab / PIVOT 시뮬레이션 — CASE+SUM 크로스탭
**EN:** A "region × month" grid (rows = region, columns = month) is built with one `SUM(CASE WHEN month = 'X' THEN amount ELSE 0 END)` per desired column — Chapter 5's conditional aggregation, just repeated once per column. BigQuery has a native `PIVOT` keyword too, but `CASE`+`SUM` works on every SQL engine and lets you name columns however you like.  
**KR:** "지역 × 월" 그리드(행=지역, 열=월)는 원하는 열마다 `SUM(CASE WHEN month = 'X' THEN amount ELSE 0 END)`를 하나씩 써서 만듭니다 — 5장의 조건부 집계를 열 하나당 한 번씩 반복하는 것뿐입니다. BigQuery에는 네이티브 `PIVOT` 키워드도 있지만, `CASE`+`SUM`은 모든 SQL 엔진에서 동작하고 열 이름을 원하는 대로 지정할 수 있습니다.

In [4]:
monthly_sales = pd.DataFrame({
    "region": ["서울","서울","서울","부산","부산","부산","인천","인천","인천"],
    "month":  ["2024-01","2024-02","2024-03"] * 3,
    "amount": [5200000, 4800000, 6100000, 3200000, 3800000, 2900000, 1800000, 2100000, 1600000],
})

sql = """
SELECT
    region,
    SUM(CASE WHEN month = '2024-01' THEN amount ELSE 0 END) AS jan,
    SUM(CASE WHEN month = '2024-02' THEN amount ELSE 0 END) AS feb,
    SUM(CASE WHEN month = '2024-03' THEN amount ELSE 0 END) AS mar,
    SUM(amount) AS total
FROM monthly_sales
GROUP BY region
ORDER BY total DESC
"""
display(run(sql))
# pandas equivalent: pd.pivot_table(df, values="amount", index="region", columns="month", aggfunc="sum", fill_value=0)


,region,jan,feb,mar,total
0,서울,5200000.0,4800000.0,6100000.0,16100000.0
1,부산,3200000.0,3800000.0,2900000.0,9900000.0
2,인천,1800000.0,2100000.0,1600000.0,5500000.0


## Example 3 — Cohort Analysis Basics: Signup-Month Retention / 코호트 분석 기초 — 가입월 기준 리텐션
**EN:** A retention cohort answers "of the people who signed up in month X, what fraction were still ordering 1, 2, 3 months later?" It's built in three named stages, each one a CTE: (1) tag each customer with their signup month, (2) tag each *order* with how many months after signup it happened, (3) count distinct customers per signup-cohort per months-since-signup, and divide by the cohort's original size.  
**KR:** 리텐션 코호트는 "X월에 가입한 사람들 중, 1개월·2개월·3개월 후에도 주문한 비율은?"에 답합니다. 이름 붙인 세 단계, 각각 CTE로 만듭니다: (1) 각 고객에게 가입월 태그, (2) 각 *주문*에 가입 후 몇 개월째인지 태그, (3) 가입 코호트 × 가입후경과월별로 고유 고객 수를 세고 코호트 원래 크기로 나눔.

In [5]:
customers3 = pd.DataFrame({
    "customer_id":  ["C01", "C02", "C03", "C04"],
    "signup_date":  ["2024-01-05", "2024-01-18", "2024-02-01", "2024-02-25"],
})
customers3["signup_date"] = pd.to_datetime(customers3["signup_date"]).dt.date

orders3 = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C01", "C02", "C01", "C04"],
    "order_date":  ["2024-01-10", "2024-01-22", "2024-02-05", "2024-02-14", "2024-03-08", "2024-03-20", "2024-02-28"],
})
orders3["order_date"] = pd.to_datetime(orders3["order_date"]).dt.date

sql = """
WITH cohort_base AS (
    -- Step 1: tag each customer with their signup month
    -- 1단계: 각 고객에게 가입월 태그
    SELECT customer_id, date_trunc('month', signup_date) AS cohort_month
    FROM customers3
),
order_with_period AS (
    -- Step 2: tag each order with months-since-signup (0 = signup month itself)
    -- 2단계: 각 주문에 가입 후 경과 개월 수 태그 (0 = 가입월 자신)
    SELECT
        o.customer_id,
        c.cohort_month,
        date_diff('month', c.cohort_month, date_trunc('month', o.order_date)) AS period_number
    FROM orders3 o
    JOIN cohort_base c ON o.customer_id = c.customer_id
),
cohort_size AS (
    SELECT cohort_month, COUNT(DISTINCT customer_id) AS cohort_customers
    FROM cohort_base GROUP BY cohort_month
),
retention_counts AS (
    -- Step 3: count distinct customers active at each period, per cohort
    -- 3단계: 코호트별로 각 시점에 활동한 고유 고객 수
    SELECT
        cohort_month,
        COUNT(DISTINCT CASE WHEN period_number = 0 THEN customer_id END) AS p0,
        COUNT(DISTINCT CASE WHEN period_number = 1 THEN customer_id END) AS p1,
        COUNT(DISTINCT CASE WHEN period_number = 2 THEN customer_id END) AS p2
    FROM order_with_period GROUP BY cohort_month
)
SELECT
    r.cohort_month, cs.cohort_customers, r.p0, r.p1, r.p2,
    ROUND(SAFE_DIVIDE(r.p1, cs.cohort_customers) * 100, 0) AS retention_m1,
    ROUND(SAFE_DIVIDE(r.p2, cs.cohort_customers) * 100, 0) AS retention_m2
FROM retention_counts r
JOIN cohort_size cs ON r.cohort_month = cs.cohort_month
ORDER BY r.cohort_month
"""
display(run(sql))
# 2024-01 cohort (C01, C02): both ordered in month 0, only C01 came back in month 1 (50%),
#   but both came back by month 2 (100%).
# 2024-02 cohort (C03, C04): ordered in month 0 only -- 0% retention so far.
# 2024-01 코호트(C01,C02): 둘 다 0개월차에 주문, 1개월차엔 C01만 재주문(50%),
#   2개월차엔 둘 다 재주문(100%).
# 2024-02 코호트(C03,C04): 0개월차에만 주문 -- 현재까지 리텐션 0%.


,cohort_month,cohort_customers,p0,p1,p2,retention_m1,retention_m2
0,2024-01-01,2,2,1,2,50.0,100.0
1,2024-02-01,2,2,0,0,0.0,0.0


## Example 4 — Deduplication: Keep Only the Latest Record / 중복 제거 — 최신 1건만 유지
**EN:** When the same real-world entity got recorded multiple times (a customer synced from an API three separate times, say), `ROW_NUMBER() OVER (PARTITION BY key ORDER BY timestamp DESC) = 1` inside `QUALIFY` keeps exactly one row per key — whichever one you sorted to the top. Simple `DISTINCT` can't do this, since it has no way to *choose which* duplicate to keep.  
**KR:** 같은 실제 개체가 여러 번 기록됐을 때(예: API에서 세 번 따로 동기화된 고객), `QUALIFY` 안의 `ROW_NUMBER() OVER (PARTITION BY key ORDER BY timestamp DESC) = 1`이 키당 정확히 한 행만 남깁니다 — 정렬해서 맨 위로 올린 그 행입니다. 단순 `DISTINCT`는 *어느 중복을* 남길지 고를 방법이 없어서 이걸 할 수 없습니다.

In [6]:
customers_raw = pd.DataFrame({
    "row_id":      [1, 2, 3, 4, 5],
    "customer_id": ["C01", "C02", "C01", "C03", "C01"],
    "name":        ["김민수", "이영희", "김민수", "박준호", "김민수"],
    "email":       ["kim@old.com", "lee@email.com", "kim@new.com", "park@email.com", "kim@new.com"],
    "updated_at":  ["2024-01-10", "2024-01-15", "2024-03-22", "2024-02-05", "2024-05-01"],
})
# C01 (김민수) is recorded 3 times, with the email changing over time / C01(김민수)이 3번 기록, 이메일이 시간에 따라 바뀜

print("-- always audit BEFORE deleting: which keys actually have duplicates? --")
print("-- 삭제 전에 항상 먼저 감사: 실제로 중복이 있는 키가 무엇인지 --")
display(run("SELECT customer_id, COUNT(*) AS cnt FROM customers_raw GROUP BY customer_id HAVING COUNT(*) > 1"))

print("-- keep only the most recently updated row per customer_id --")
print("-- 고객별로 가장 최근에 업데이트된 행만 유지 --")
sql = """
SELECT * EXCLUDE(row_id)
FROM customers_raw
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) = 1
"""
display(run(sql))
# 김민수's row now shows the newest email (kim@new.com, from 2024-05-01) -- the two older,
# now-outdated copies are gone.
# 김민수 행이 이제 최신 이메일(kim@new.com, 2024-05-01)을 보여줌 -- 오래되고
# 이제는 낡은 두 개의 복사본은 사라짐.
# (Note: BigQuery spells this SELECT * EXCEPT(row_id); this notebook's engine (DuckDB) uses EXCLUDE.)
# (참고: BigQuery는 SELECT * EXCEPT(row_id)로 씁니다; 이 노트북의 엔진(DuckDB)은 EXCLUDE를 씁니다.)


-- always audit BEFORE deleting: which keys actually have duplicates? --
-- 삭제 전에 항상 먼저 감사: 실제로 중복이 있는 키가 무엇인지 --


,customer_id,cnt
0,C01,3


-- keep only the most recently updated row per customer_id --
-- 고객별로 가장 최근에 업데이트된 행만 유지 --


,customer_id,name,email,updated_at
0,C01,김민수,kim@new.com,2024-05-01
1,C02,이영희,lee@email.com,2024-01-15
2,C03,박준호,park@email.com,2024-02-05


## Example 5 — Data Quality Check Queries / 데이터 품질 체크 쿼리
**EN:** Before trusting any new table, run three quick checks: (1) `NULL` rates per column, using `COUNTIF`/`COUNT` from Chapter 5; (2) exact duplicate rows, using `COUNT(*)` vs. `COUNT(DISTINCT id)` from Chapter 2; (3) numeric outliers, using the IQR method — values far outside the 25th–75th percentile range.  
**KR:** 새로운 테이블을 신뢰하기 전에 세 가지를 빠르게 점검하세요: (1) 5장의 `COUNTIF`/`COUNT`로 열별 `NULL` 비율, (2) 2장의 `COUNT(*)` vs `COUNT(DISTINCT id)`로 완전 중복 행, (3) IQR(사분위범위) 방식으로 숫자 이상치 — 25~75 백분위 범위를 크게 벗어난 값들.

In [7]:
orders_raw = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1005, 1005, 1006],
    "customer_id": ["C01", None, "C01", "C03", "C02", "C02", "C01"],
    "amount":      [45000, 32000, None, 28000, 95000, 95000, 9999999],   # a NULL amount, and a wild outlier
    "status":      ["완료", "완료", "취소", "완료", "완료", "완료", "완료"],
})
# order 1005 is duplicated, and order 1006's amount (9,999,999) looks like a data-entry error.
# 1005 주문은 중복이고, 1006 주문의 금액(9,999,999)은 입력 오류처럼 보임.

print("-- Check 1: NULL rates + duplicate row count, all in one query --")
print("-- 체크 1: NULL 비율 + 중복 행 수, 한 쿼리로 --")
sql1 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_rows,
    COUNTIF(customer_id IS NULL) AS null_customer_id,
    COUNTIF(amount IS NULL) AS null_amount,
    ROUND(COUNTIF(amount IS NULL) / COUNT(*) * 100, 1) AS null_amount_pct
FROM orders_raw
"""
display(run(sql1))

print("-- Check 2: outliers via IQR (values outside 1.5x the interquartile range) --")
print("-- 체크 2: IQR 기반 이상치 (사분위범위의 1.5배를 벗어난 값) --")
sql2 = """
WITH percentiles AS (
    SELECT
        quantile_cont(amount, 0.25) OVER () AS q1,
        quantile_cont(amount, 0.75) OVER () AS q3
    FROM orders_raw WHERE amount IS NOT NULL LIMIT 1
),
bounds AS (
    SELECT q1, q3,
        q1 - 1.5 * (q3 - q1) AS lower_fence,
        q3 + 1.5 * (q3 - q1) AS upper_fence
    FROM percentiles
)
SELECT o.order_id, o.amount, b.lower_fence, b.upper_fence,
    CASE
        WHEN o.amount < b.lower_fence THEN '하한 이상치'
        WHEN o.amount > b.upper_fence THEN '상한 이상치'
        ELSE '정상'
    END AS outlier_flag
FROM orders_raw o
CROSS JOIN bounds b
WHERE o.amount IS NOT NULL
ORDER BY o.amount DESC
"""
display(run(sql2))
# order 1006 (9,999,999) is flagged as an upper-bound outlier -- almost certainly a data-entry mistake.
# (this notebook's engine: BigQuery's PERCENTILE_CONT is DuckDB's quantile_cont -- same idea, different name)
# 1006 주문(9,999,999)이 상한 이상치로 표시됨 -- 거의 확실히 입력 실수.
# (이 노트북의 엔진: BigQuery의 PERCENTILE_CONT는 DuckDB의 quantile_cont -- 같은 개념, 다른 이름)


-- Check 1: NULL rates + duplicate row count, all in one query --
-- 체크 1: NULL 비율 + 중복 행 수, 한 쿼리로 --


,total_rows,duplicate_rows,null_customer_id,null_amount,null_amount_pct
0,7,1,1.0,1.0,14.3


-- Check 2: outliers via IQR (values outside 1.5x the interquartile range) --
-- 체크 2: IQR 기반 이상치 (사분위범위의 1.5배를 벗어난 값) --


,order_id,amount,lower_fence,upper_fence,outlier_flag
0,1006,9999999.0,-54375.0,184625.0,상한 이상치
1,1005,95000.0,-54375.0,184625.0,정상
2,1005,95000.0,-54375.0,184625.0,정상
3,1001,45000.0,-54375.0,184625.0,정상
4,1002,32000.0,-54375.0,184625.0,정상
5,1004,28000.0,-54375.0,184625.0,정상


## Example 6 — Common Combinations / 자주 쓰는 조합
**EN:** **Pattern A** chains 3 CTEs into a complete monthly dashboard in one query: base metrics → MoM % → cumulative total — each stage independently testable by running just that CTE with `SELECT * FROM stage_name`. **Pattern B** builds a full customer profile by joining a purchase-summary CTE back to the customer table with `LEFT JOIN` (so non-buyers survive as `0`, not disappear), then applies a `CASE WHEN` tier on top.  
**KR:** **패턴 A**는 CTE 3개를 이어서 한 쿼리로 완전한 월별 대시보드를 만듭니다: 기본 지표 → MoM % → 누적 합계 — 각 단계는 `SELECT * FROM 단계이름`으로 그 CTE만 실행해보면 독립적으로 테스트 가능합니다. **패턴 B**는 구매 요약 CTE를 `LEFT JOIN`으로 고객 테이블에 다시 결합해서(비구매자도 `0`으로 살아남고 사라지지 않음) 완전한 고객 프로필을 만든 뒤, 그 위에 `CASE WHEN` 등급을 적용합니다.

In [8]:
print("-- Pattern A: 3-CTE monthly dashboard chain / 3-CTE 월별 대시보드 체인 --")
orders_a = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1005, 1006],
    "customer_id": ["C01", "C02", "C01", "C03", "C02", "C01"],
    "order_date":  ["2024-01-15", "2024-01-22", "2024-02-05", "2024-02-18", "2024-03-01", "2024-03-15"],
    "amount":      [45000, 32000, 61000, 28000, 95000, 42000],
    "status":      ["완료", "완료", "완료", "취소", "완료", "완료"],
})
orders_a["order_date"] = pd.to_datetime(orders_a["order_date"]).dt.date
sql_a = """
WITH monthly_base AS (
    SELECT
        date_trunc('month', order_date) AS month,
        COUNT(*) AS total_orders,
        COUNTIF(status = '완료') AS completed_orders,
        SUM(CASE WHEN status = '완료' THEN amount ELSE 0 END) AS revenue
    FROM orders_a
    GROUP BY month
),
with_mom AS (
    SELECT month, revenue, completed_orders,
        ROUND(SAFE_DIVIDE(
            revenue - LAG(revenue) OVER (ORDER BY month),
            LAG(revenue) OVER (ORDER BY month)
        ) * 100, 1) AS mom_pct
    FROM monthly_base
),
final AS (
    SELECT month, revenue, mom_pct,
        SUM(revenue) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_revenue
    FROM with_mom
)
SELECT
    strftime(month, '%Y-%m') AS month,
    revenue,
    CASE WHEN mom_pct IS NULL THEN 'N/A' ELSE CONCAT(CAST(mom_pct AS VARCHAR), '%') END AS mom,
    cumulative_revenue
FROM final
ORDER BY month
"""
display(run(sql_a))

print("-- Pattern B: customer segment + purchase stats / 고객 세그먼트 + 구매 통계 --")
customers_b = pd.DataFrame({"customer_id": ["C01","C02","C03"], "name": ["김민수","이영희","박준호"], "signup_date": ["2023-01-10","2023-06-15","2024-02-01"]})
customers_b["signup_date"] = pd.to_datetime(customers_b["signup_date"]).dt.date
orders_b = pd.DataFrame({
    "order_id": [1001,1002,1003,1004,1005],
    "customer_id": ["C01","C01","C02","C02","C03"],
    "amount": [45000,61000,32000,95000,28000],
    "order_date": ["2024-01-15","2024-02-03","2024-01-22","2024-03-10","2024-03-25"],
})
orders_b["order_date"] = pd.to_datetime(orders_b["order_date"]).dt.date
sql_b = """
WITH customer_orders AS (
    SELECT customer_id, COUNT(*) AS order_count, SUM(amount) AS total_spent, MAX(order_date) AS last_order_date
    FROM orders_b GROUP BY customer_id
),
customer_profile AS (
    SELECT
        c.customer_id, c.name,
        date_diff('day', c.signup_date, DATE '2024-06-01') AS days_since_signup,
        COALESCE(co.order_count, 0) AS order_count,
        COALESCE(co.total_spent, 0) AS total_spent,
        co.last_order_date
    FROM customers_b c
    LEFT JOIN customer_orders co ON c.customer_id = co.customer_id
)
SELECT
    name, days_since_signup, order_count, total_spent, last_order_date,
    CASE
        WHEN total_spent >= 100000 THEN 'Gold'
        WHEN total_spent >= 50000  THEN 'Silver'
        WHEN order_count = 0       THEN '미구매'
        ELSE 'Bronze'
    END AS tier
FROM customer_profile
ORDER BY total_spent DESC
"""
display(run(sql_b))


-- Pattern A: 3-CTE monthly dashboard chain / 3-CTE 월별 대시보드 체인 --


,month,revenue,mom,cumulative_revenue
0,2024-01,77000.0,N/A,77000.0
1,2024-02,61000.0,-20.8%,138000.0
2,2024-03,137000.0,124.6%,275000.0


-- Pattern B: customer segment + purchase stats / 고객 세그먼트 + 구매 통계 --


,name,days_since_signup,order_count,total_spent,last_order_date,tier
0,이영희,352,2,127000.0,2024-03-10,Gold
1,김민수,508,2,106000.0,2024-02-03,Gold
2,박준호,121,1,28000.0,2024-03-25,Bronze


## Example 7 — Practice: Integrated 3-Step CTE / 실습 문제 — 통합 (CTE 3단계)
**EN:** This final exercise combines the whole guide: `JOIN` (Ch.3) + `GROUP BY` (Ch.2) → Pareto window functions (this chapter) → `RANK` within category (Ch.7). Fill in each `________` blank, then remove the `#` to check your answer. Hints: `order_items` / `JOIN` / `UNBOUNDED PRECEDING` / `CURRENT ROW` / `RANK` / `DESC`  
**KR:** 이 마지막 연습 문제는 가이드 전체를 조합합니다: `JOIN`(3장) + `GROUP BY`(2장) → 파레토 윈도우 함수(이번 챕터) → 카테고리 내 `RANK`(7장). `________` 빈칸을 채운 뒤 `#`을 지워서 답을 확인하세요. 힌트: `order_items` / `JOIN` / `UNBOUNDED PRECEDING` / `CURRENT ROW` / `RANK` / `DESC`

In [11]:
products_p = pd.DataFrame({
    "product_id": ["P01", "P02", "P03", "P04", "P05"],
    "name":       ["노트북", "마우스", "키보드", "셔츠", "청바지"],
    "category":   ["전자", "전자", "전자", "의류", "의류"],
    "price":      [1200000, 25000, 45000, 35000, 89000],
})
order_items_p = pd.DataFrame({
    "order_id":   [5001, 5002, 5003, 5004, 5005, 5006],
    "product_id": ["P01", "P02", "P01", "P03", "P05", "P02"],
    "quantity":   [1, 3, 2, 1, 2, 5],
})
# Note: P04 (셔츠) never appears in order_items_p / P04(셔츠)는 order_items_p에 한 번도 등장하지 않음

query = """
-- Step 1: per-product revenue and quantity sold (JOIN + GROUP BY)
WITH product_revenue AS (
    SELECT
        oi.product_id,
        p.name,
        p.category,
        SUM(oi.quantity * p.price) AS revenue,
        SUM(oi.quantity) AS total_qty
    FROM order_items_p oi
    JOIN products_p p ON oi.product_id = p.product_id
    GROUP BY oi.product_id, p.name, p.category
),
-- Step 2: Pareto -- % of total, and cumulative %
with_pareto AS (
    SELECT
        product_id, name, category, revenue, total_qty,
        ROUND(revenue / SUM(revenue) OVER () * 100, 1) AS rev_pct,
        ROUND(
            SUM(revenue) OVER (ORDER BY revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
            / SUM(revenue) OVER () * 100, 1
        ) AS cum_pct
    FROM product_revenue
),
-- Step 3: rank within each category
final AS (
    SELECT
        name, category, revenue, rev_pct, cum_pct,
        RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS rank_in_cat
    FROM with_pareto
)
SELECT * FROM final ORDER BY revenue DESC
"""
display(run(query))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

print("✏️  Fill in the ________ blanks above, uncomment the display() line, then re-run this cell.")
print("✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.")


,name,category,revenue,rev_pct,cum_pct,rank_in_cat
0,노트북,전자,3600000.0,89.5,89.5,1
1,마우스,전자,200000.0,5.0,94.5,2
2,청바지,의류,178000.0,4.4,98.9,1
3,키보드,전자,45000.0,1.1,100.0,3


✏️  Fill in the ________ blanks above, uncomment the display() line, then re-run this cell.
✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.


<details>
<summary>🔑 Answer / 정답 (click to expand / 클릭해서 펼치기)</summary>

```sql
WITH product_revenue AS (
    SELECT
        oi.product_id, p.name, p.category,
        SUM(oi.quantity * p.price) AS revenue,
        SUM(oi.quantity) AS total_qty
    FROM order_items_p oi
    JOIN products_p p ON oi.product_id = p.product_id
    GROUP BY oi.product_id, p.name, p.category
),
with_pareto AS (
    SELECT
        product_id, name, category, revenue, total_qty,
        ROUND(revenue / SUM(revenue) OVER () * 100, 1) AS rev_pct,
        ROUND(
            SUM(revenue) OVER (ORDER BY revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
            / SUM(revenue) OVER () * 100, 1
        ) AS cum_pct
    FROM product_revenue
),
final AS (
    SELECT
        name, category, revenue, rev_pct, cum_pct,
        RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS rank_in_cat
    FROM with_pareto
)
SELECT * FROM final ORDER BY revenue DESC
```

**Reading the result / 결과 읽기:** 노트북 alone is ~89% of revenue (Pareto, extreme concentration); within the 전자 category it's rank 1, 마우스 rank 2, 키보드 rank 3; 셔츠 doesn't appear at all — it was never ordered, so the `JOIN` (not `LEFT JOIN`) drops it. 노트북 하나가 매출의 약 89%(극단적인 파레토 집중); 전자 카테고리 안에서 노트북 1위, 마우스 2위, 키보드 3위; 셔츠는 전혀 안 나옴 — 한 번도 주문되지 않아 `JOIN`(`LEFT JOIN`이 아니므로)이 제외함.
</details>

---
# ⚠️ Common Mistakes

**Mistake 1 — Trusting `LAG(revenue, 12)` for YoY without checking for gaps in the data**
- EN: `LAG(col, 12)` reaches back 12 *rows*, not 12 calendar months. If even one month is missing from the underlying data, every YoY comparison after that gap silently lines up against the wrong month.
- KR: `LAG(열, 12)`는 12 *행* 뒤를 보는 것이지, 12 캘린더 월이 아닙니다. 원본 데이터에 단 한 달이라도 빠지면, 그 빈틈 이후의 모든 YoY 비교가 조용히 엉뚱한 달과 비교됩니다.
- ✅ Fix / 해결법: Verify the data has zero missing months before trusting `LAG(12)`, or use a more robust `Self JOIN` + `DATE_DIFF(..., MONTH) = 12` approach that doesn't depend on row position.  
`LAG(12)`를 신뢰하기 전에 빠진 달이 없는지 확인하거나, 행 위치에 의존하지 않는 `Self JOIN` + `DATE_DIFF(..., MONTH) = 12` 방식을 쓰세요.

**Mistake 2 — Deleting "duplicates" without auditing which rows they actually are first**
- EN: What looks like a duplicate can sometimes be a legitimate repeat event (a customer genuinely placing two identical orders). Running `ROW_NUMBER() ... = 1` straight away, without first looking at *what* the duplicate rows contain, risks silently discarding real data.
- KR: 중복처럼 보이는 것이 실제로는 정당한 반복 이벤트(고객이 정말로 똑같은 주문을 두 번 넣은 경우)일 수 있습니다. 중복 행이 *무엇을* 담고 있는지 먼저 보지 않고 바로 `ROW_NUMBER() ... = 1`을 실행하면 실제 데이터를 조용히 버릴 위험이 있습니다.
- ✅ Fix / 해결법: Always run the audit query (`GROUP BY key HAVING COUNT(*) > 1`) first, and manually inspect a sample of what's being called a "duplicate" before removing anything.  
항상 감사 쿼리(`GROUP BY key HAVING COUNT(*) > 1`)를 먼저 실행하고, 무언가를 지우기 전에 "중복"이라 불리는 것의 샘플을 직접 확인하세요.

**Mistake 3 — Debugging a long CTE chain from the final SELECT backward**
- EN: When a 3-4 stage CTE chain produces a wrong final number, the instinct is to stare at the last `SELECT` — but the bug is just as likely to be in an earlier stage that silently fed bad data forward.
- KR: 3~4단계 CTE 체인이 틀린 최종 숫자를 낼 때, 본능적으로 마지막 `SELECT`를 들여다보게 되지만, 버그는 그만큼이나 자주 앞쪽 단계에 있어서 조용히 잘못된 데이터를 다음으로 넘긴 것일 수 있습니다.
- ✅ Fix / 해결법: Debug from the *first* CTE forward — run `SELECT * FROM cohort_base` (etc.) on its own for each stage in order, confirming each is correct before trusting the next one built on top of it.  
*첫* CTE부터 순서대로 디버깅하세요 — 각 단계마다 `SELECT * FROM cohort_base`처럼 단독으로 실행해서, 그 위에 쌓은 다음 단계를 신뢰하기 전에 각각이 맞는지 확인하세요.

**Mistake 4 — Skipping the data quality check because the table "looks fine" in a preview**
- EN: A quick `SELECT * LIMIT 10` preview can look completely clean while the full table hides a 15% `NULL` rate in a column you didn't scroll to, or a handful of catastrophic outliers buried in millions of normal rows.
- KR: `SELECT * LIMIT 10` 미리보기는 완전히 깨끗해 보일 수 있지만, 전체 테이블에는 스크롤하지 않은 열에 15%의 `NULL` 비율이 숨어 있거나, 수백만 개의 정상 행 속에 극단적인 이상치 몇 개가 묻혀 있을 수 있습니다.
- ✅ Fix / 해결법: Run the 3-part quality check (`NULL` rates, duplicate count, outlier scan) on every new table before building anything on top of it — it takes one query and a few seconds.  
새로운 테이블 위에 무언가를 쌓기 전에 항상 3단계 품질 체크(`NULL` 비율, 중복 개수, 이상치 스캔)를 실행하세요 — 쿼리 하나, 몇 초면 됩니다.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- Before writing a single line of SQL for a new business question, name the *pattern* first ("this is a MoM report," "this is a dedup," "this is a cohort") — the pattern tells you which tools from this chapter to reach for.  
 새로운 비즈니스 질문에 SQL 한 줄이라도 쓰기 전에, 먼저 *패턴*의 이름을 붙여보세요("이건 MoM 리포트다", "이건 dedup이다", "이건 코호트다") — 그 패턴이 이번 챕터의 어떤 도구를 꺼내야 할지 알려줍니다.
- Build long CTE chains incrementally: write and test CTE 1 alone, confirm it's right, *then* add CTE 2 on top — never write all 3-4 stages blind and debug the whole thing at once.  
 긴 CTE 체인은 점진적으로 만드세요: CTE 1만 먼저 작성하고 테스트해서 맞는지 확인한 *다음* CTE 2를 그 위에 추가하세요 — 3~4단계를 한꺼번에 눈감고 작성해서 통째로 디버깅하지 마세요.
- Keep a personal "data quality check" query template saved somewhere — you'll run some version of it at the start of nearly every new analysis for the rest of your career.  
 개인용 "데이터 품질 체크" 쿼리 템플릿을 어딘가에 저장해 두세요 — 앞으로 커리어 내내 거의 모든 새 분석을 시작할 때마다 이걸 어떤 형태로든 실행하게 될 것입니다.
- When a stakeholder's question sounds like "are we doing better than before," your ears should immediately perk up for `LAG` + `DATE_TRUNC` — it's the single most re-used pattern in a BA's day-to-day work.  
 이해관계자의 질문이 "예전보다 잘하고 있나요"처럼 들리면, 즉시 `LAG` + `DATE_TRUNC`를 떠올리세요 — BA의 일상 업무에서 가장 많이 재사용되는 패턴입니다.

---
# 🔗 Related Concepts

```
SQL Learning Roadmap (this guide) — COMPLETE / SQL 학습 로드맵 (이 가이드) — 완주
──────────────────────────────────────────────
 1. SELECT Basics                 ─┐
 2. Aggregation & GROUP BY         │
 3. JOIN                           │  all seven of these tools
 4. Subquery & CTE                 │  combine in THIS chapter
 5. Conditions & NULL Handling     │  이 7개 도구가 전부
 6. String & Date Functions        │  이번 챕터에서 조합됨
 7. Window Functions              ─┘
 8. BA-Specific Patterns          ← ★ YOU ARE HERE / 지금 여기 (final / 마지막)
```

```
Which chapters feed each pattern here / 각 패턴이 어느 챕터에서 왔는지
──────────────────────────────────────────────
  MoM / YoY            <- Ch.7 (LAG) + Ch.6 (DATE_TRUNC/DATE_DIFF)
  Pareto analysis       <- Ch.7 (running SUM OVER)
  PIVOT simulation      <- Ch.2 (GROUP BY) + Ch.5 (CASE WHEN)
  Cohort analysis       <- Ch.4 (CTE) + Ch.3 (JOIN) + Ch.6 (dates) + Ch.2 (GROUP BY)
  Deduplication         <- Ch.7 (ROW_NUMBER/QUALIFY)
  Data quality checks   <- Ch.2 (COUNT) + Ch.5 (COUNTIF) + Ch.7 (quantile window)
```

*How is today's topic connected to other concepts?*

**EN:** This chapter doesn't introduce anything new to connect *forward* to — it's the connection point for everything that came *before*. If any pattern here felt unfamiliar, that's a signal pointing back to a specific earlier chapter (the roadmap above shows exactly which one) rather than something to memorize fresh. That's the real shape of SQL fluency: not knowing more keywords, but recognizing which small set of tools you already have is the right combination for the business question in front of you.

**KR:** 이번 챕터는 *앞으로* 연결될 새로운 것을 소개하지 않습니다 — *이전에* 배운 모든 것이 만나는 지점입니다. 여기서 어떤 패턴이 낯설게 느껴졌다면, 그건 새로 외워야 할 것이 아니라 특정 이전 챕터를 다시 봐야 한다는 신호입니다(위 로드맵이 정확히 어느 챕터인지 보여줍니다). 이것이 SQL 능숙함의 진짜 모습입니다: 더 많은 키워드를 아는 게 아니라, 이미 가진 작은 도구 세트 중 무엇이 눈앞의 비즈니스 질문에 맞는 조합인지 알아보는 것입니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오:**
**EN:** It's the start of the month, and your manager sends one message covering three separate asks: *"For the leadership deck — (1) are we growing month over month, (2) which products are we over-indexed on for revenue, and (3) can you double check this new data feed doesn't have any obvious garbage in it before I present any of it?"* This is three patterns from this chapter, back to back.
**KR:** 월초, 매니저가 메시지 하나로 세 가지를 한꺼번에 요청합니다: *"리더십 덱용으로 — (1) 우리 전월 대비 성장하고 있는지, (2) 매출이 특정 제품에 쏠려 있는지, (3) 발표하기 전에 이 새 데이터 피드에 명백한 쓰레기 데이터가 없는지 확인해줄 수 있어?"* 이번 챕터의 세 패턴이 연달아 나옵니다.

**To-do / 할 일:**
- [x] Quality-check the new feed first — before anything else gets built on top of it  
다른 무엇을 쌓기 전에 새 피드를 먼저 품질 체크한다
- [x] Build the MoM growth number  
MoM 성장률 숫자를 만든다
- [x] Build the Pareto breakdown for revenue concentration  
매출 집중도를 위한 파레토 분석을 만든다

In [10]:
# Step 1: quality check the new feed / 새 피드 품질 체크
new_feed = pd.DataFrame({
    "order_id": [9001, 9002, 9003, 9004, 9005],
    "customer_id": ["C01", "C02", None, "C03", "C01"],
    "amount": [52000, 38000, 61000, 45000, 8888888],
})
print("-- quality check / 품질 체크 --")
display(run("""
SELECT COUNT(*) AS total_rows, COUNTIF(customer_id IS NULL) AS null_customers,
       MAX(amount) AS max_amount, MIN(amount) AS min_amount
FROM new_feed
"""))
print("⚠️  max_amount (8,888,888) looks like an outlier relative to the rest -- flag before presenting.")
print("⚠️  max_amount(8,888,888)이 나머지에 비해 이상치로 보임 -- 발표 전에 표시해둘 것.")

# Step 2: MoM growth / MoM 성장률
print()
print("-- MoM growth / MoM 성장률 --")
mr_biz = pd.DataFrame({"month": ["2024-04","2024-05","2024-06"], "revenue": [61000000, 58000000, 67000000]})
display(run("""
SELECT month, revenue,
    ROUND(SAFE_DIVIDE(revenue - LAG(revenue) OVER (ORDER BY month), LAG(revenue) OVER (ORDER BY month)) * 100, 1) AS mom_pct
FROM mr_biz ORDER BY month
"""))


-- quality check / 품질 체크 --


,total_rows,null_customers,max_amount,min_amount
0,5,1.0,8888888,38000


⚠️  max_amount (8,888,888) looks like an outlier relative to the rest -- flag before presenting.
⚠️  max_amount(8,888,888)이 나머지에 비해 이상치로 보임 -- 발표 전에 표시해둘 것.

-- MoM growth / MoM 성장률 --


,month,revenue,mom_pct
0,2024-04,61000000,NaN
1,2024-05,58000000,-4.9
2,2024-06,67000000,15.5


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**EN:** This chapter's six patterns are all recombinations of Chapters 1–7's tools: MoM/YoY growth uses `LAG` plus date functions (and `LAG(col, 12)` is fragile without guaranteed-complete monthly data); Pareto analysis uses a running-total window function divided by the grand total; a PIVOT crosstab is repeated conditional `SUM`/`CASE WHEN` calls, one per output column; cohort retention chains multiple CTEs, each tagging and counting a different slice of the data; deduplication uses `ROW_NUMBER` + `QUALIFY` to keep exactly one "winning" row per key, after first auditing what the duplicates actually are; and a data-quality check combines `NULL` rate checks, exact-duplicate counts, and IQR-based outlier detection into one pre-analysis ritual. None of these are new syntax — they're recognizable shapes worth memorizing as units, since real business questions arrive already shaped like one of them.

**KR:** 이번 챕터의 여섯 패턴은 모두 1~7장 도구의 재조합입니다: MoM/YoY 성장률은 `LAG`와 날짜 함수를 쓰고(`LAG(col, 12)`는 빠짐없는 완전한 월별 데이터가 보장되지 않으면 취약함), 파레토 분석은 전체 합계로 나눈 누적합 윈도우 함수를 쓰며, PIVOT 크로스탭은 출력 열마다 하나씩 반복되는 조건부 `SUM`/`CASE WHEN` 호출이고, 코호트 리텐션은 여러 CTE를 이어서 각각 데이터의 다른 조각에 태그를 붙이고 세며, 중복 제거는 먼저 중복이 실제로 무엇인지 감사한 뒤 `ROW_NUMBER` + `QUALIFY`로 키당 정확히 하나의 "승리한" 행만 남기고, 데이터 품질 체크는 `NULL` 비율 확인·정확한 중복 개수·IQR 기반 이상치 탐지를 분석 전 의식 하나로 합칩니다. 이 중 새로운 문법은 하나도 없습니다 — 실제 비즈니스 질문이 이미 이 중 하나의 모양으로 도착하므로, 단위로 외워둘 가치가 있는 알아볼 수 있는 형태들입니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence. / 오늘 배운 내용을 한 문장으로.

> **EN:** Every pattern in this final chapter — and honestly, in this entire guide — comes down to the same move: name what the business question actually needs (a comparison, a running share, a reshaped grid, a per-group latest record, a trust check), then reach for the one or two tools from Chapters 1–7 built exactly for that need.

> **KR:** 이 마지막 챕터의, 그리고 솔직히 이 가이드 전체의 모든 패턴은 결국 같은 동작으로 귀결됩니다: 비즈니스 질문이 실제로 필요로 하는 것(비교, 누적 비율, 재구성된 그리드, 그룹별 최신 기록, 신뢰성 체크)의 이름을 먼저 붙이고, 정확히 그 필요를 위해 만들어진 1~7장의 도구 한두 개를 꺼내드는 것입니다.

---
# ❓ Review Questions

**Q1.** Why is `LAG(revenue, 12)` a risky way to compute YoY growth if you can't guarantee the underlying monthly data has zero gaps?
**Q1.** 원본 월별 데이터에 빈 달이 전혀 없다고 보장할 수 없다면, 왜 `LAG(revenue, 12)`로 YoY 성장률을 계산하는 것이 위험한가?

**Q2.** In a Pareto analysis, what two numbers do you divide to get `cum_pct`, and why must the data be sorted by `revenue DESC` first for that number to mean anything?
**Q2.** 파레토 분석에서 `cum_pct`를 구하려면 어떤 두 숫자를 나누는가, 그리고 그 숫자가 의미를 가지려면 왜 데이터가 먼저 `revenue DESC`로 정렬되어야 하는가?

**Q3.** Walk through the 3-CTE cohort analysis: what does each of `cohort_base`, `order_with_period`, and `retention_counts` compute, in your own words?
**Q3.** 3-CTE 코호트 분석을 따라가 보라: `cohort_base`, `order_with_period`, `retention_counts` 각각이 무엇을 계산하는지 자신의 말로 설명하라.

**Q4.** Before running a `ROW_NUMBER` + `QUALIFY` deduplication query, what should you check first, and why?
**Q4.** `ROW_NUMBER` + `QUALIFY` 중복 제거 쿼리를 실행하기 전에 먼저 무엇을 확인해야 하며, 왜 그런가?

**Q5.** Looking back across all 8 chapters, which single concept from an earlier chapter do you feel least confident about — and which example in *this* chapter would be the fastest way to go practice it again?
**Q5.** 8개 챕터 전체를 돌아볼 때, 이전 챕터의 어떤 개념이 가장 자신 없게 느껴지는가 — 그리고 *이번* 챕터의 어떤 예제가 그것을 다시 연습하는 가장 빠른 방법이겠는가?

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*

---
## 🎉 Guide Complete / 가이드 완주
**EN:** That's all 8 chapters — from `SELECT * FROM orders` in Chapter 1 to a 3-stage cohort retention CTE chain here. The syntax will fade if unused; the *patterns* in this final chapter are what's worth keeping sharp through spaced repetition.
**KR:** 8개 챕터 전부 끝났습니다 — 1장의 `SELECT * FROM orders`부터 여기 3단계 코호트 리텐션 CTE 체인까지. 문법은 안 쓰면 흐려지지만, 이 마지막 챕터의 *패턴*들은 분산 반복으로 계속 날카롭게 유지할 가치가 있습니다.